## Определение объектов в видеофайле

## Шаг 1. Установка зависимостей

In [ ]:
#pip install ultralytics opencv-python numpy matplotlib

## Шаг 2. Импорт библиотек

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import csv
import os

## Шаг 3. Загрузка модели и настройка

In [2]:

# Загрузка модели YOLOv8
#model = YOLO('yolov8n.pt')  # можно заменить на yolov8s.pt, yolov8m.pt
model = YOLO("yolo26n.pt") 

# Список цветов для разных классов
colors = [
    (255, 0, 0), (0, 255, 0), (0, 0, 255),
    (255, 255, 0), (0, 255, 255), (255, 0, 255)
]

## Шаг 4. Функция для обработки кадра

In [3]:
def process_frame(frame, model, colors, confidence_threshold=0.5):
    """
    Обрабатывает один кадр: детектирует объекты и рисует рамки.
    Возвращает обработанный кадр и данные о детектированных объектах.
    """
    # Обработка кадра моделью YOLO
    results = model(frame)[0]

    # Получение данных об объектах
    classes_names = results.names
    classes = results.boxes.cls.cpu().numpy()
    boxes = results.boxes.xyxy.cpu().numpy().astype(np.int32)
    confidences = results.boxes.conf.cpu().numpy()

    detection_data = []  # список для хранения данных о детектированных объектах

    # Рисование рамок и подписей на кадре
    for class_id, box, conf in zip(classes, boxes, confidences):
        if conf > confidence_threshold:
            class_name = classes_names[int(class_id)]
            color = colors[int(class_id) % len(colors)]
            x1, y1, x2, y2 = box

            # Отрисовка рамки
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            # Отрисовка подписи
            label = f"{class_name}: {conf:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Сохранение данных о детектированном объекте
            detection_data.append({
                'class': class_name,
                'confidence': conf,
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
                'width': x2 - x1,
                'height': y2 - y1
            })

    return frame, detection_data

## Шаг 5. Основной цикл обработки видео

In [ ]:
Скачайте 

In [4]:
# Путь к видеофайлу
#input_video_path = 'input_video.mp4'  # замените на путь к вашему видео
#input_video_path = r"Мини-футбол_Чемпионат_МГУ-2023.Финал.mp4"
input_video_path = r"https://disk.yandex.ru/i/7hvOOcffkyAYPA"

# Пути для сохранения результатов
output_video_path = 'output_video_with_detection.mp4'
csv_output_path = 'detection_results.csv'

# Открытие видеофайла
cap = cv2.VideoCapture(input_video_path)

# Проверка, удалось ли открыть видео
if not cap.isOpened():
    print("Ошибка: Не удалось открыть видеофайл")
else:
    print("Видео успешно открыто. Начинаю обработку...")

# Получение параметров видео
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Создание объекта для записи видео
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Подготовка CSV‑файла для записи результатов
csv_file = open(csv_output_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.DictWriter(csv_file, fieldnames=[
    'frame_number', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2', 'width', 'height'
])
csv_writer.writeheader()

frame_count = 0  # счётчик кадров


# Основной цикл обработки
try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Достигнут конец видео или произошла ошибка при чтении кадра.")
            break

        frame_count += 1

        # Обработка кадра
        processed_frame, detection_data = process_frame(frame, model, colors)

        # Запись обработанного кадра в выходной видеофайл
        out.write(processed_frame)

        # Запись данных о детектированных объектах в CSV
        for obj in detection_data:
            obj['frame_number'] = frame_count
            csv_writer.writerow(obj)

        # Конвертация из BGR (OpenCV) в RGB (matplotlib)
        rgb_frame = cv2.cvtColor(processed_frame, cv2.COLOR_BGR2RGB)

        # Очистка предыдущего вывода
        clear_output(wait=True)

        # Отображение кадра в Jupyter
        plt.figure(figsize=(12, 8))
        plt.imshow(rgb_frame)
        plt.axis('off')  # Убираем оси
        plt.title(f'YOLO Object Detection — Кадр {frame_count}')
        plt.show()

except KeyboardInterrupt:
    print("Обработка прервана пользователем.")

finally:
    # Освобождение ресурсов
    cap.release()
    out.release()  # закрываем видеофайл для записи
    csv_file.close()  # закрываем CSV‑файл

    print(f"Обработка завершена. Ресурсы освобождены.")
    print(f"Обработанное видео сохранено как: {output_video_path}")
    print(f"Результаты детекции сохранены как: {csv_output_path}")

    # Показываем статистику
    if os.path.exists(csv_output_path):
        import pandas as pd
        results_df = pd.read_csv(csv_output_path)
        print(f"\nВсего обработано кадров: {frame_count}")
        print(f"Всего детектированных объектов: {len(results_df)}")
        print("\nСтатистика по классам:")
        print(results_df['class'].value_counts())

Ошибка: Не удалось открыть видеофайл
Обработка завершена. Ресурсы освобождены.
Обработанное видео сохранено как: output_video_with_detection.mp4
Результаты детекции сохранены как: detection_results.csv

Всего обработано кадров: 0
Всего детектированных объектов: 0

Статистика по классам:
Series([], Name: count, dtype: int64)
